# 01 - Main workflow

End-to-end reproduction of the reported model: velocity models to forward modeling to
pre-processing to U-Net training to evaluation.

Run the cells in order. Sections 1 to 3 use PyTorch and Deepwave; sections 4 onward use
TensorFlow/Keras. If the two frameworks contend for GPU memory, restart the kernel after
section 3 and reload the cached stacks from `outputs/stacked/`.

## 0. Setup and reproducibility

In [ ]:
# Global reproducibility seeds.
# Every result reported in the paper was produced with GLOBAL_SEED = 42.
import os, random
import numpy as np
import torch
import tensorflow as tf

GLOBAL_SEED = 42
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('PyTorch', torch.__version__, '| TensorFlow', tf.__version__)

for d in ['data/velocity_models', 'models', 'outputs/stacked', 'outputs/figures']:
    os.makedirs(d, exist_ok=True)

## 1. Load the CCS time-lapse velocity models

Baseline, monitoring stage 1 (reported case), and monitoring stage 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Define the file paths and their corresponding names
file_info = {
    'baseline': 'data/velocity_models/baseline.bin',
    'monitoring_stage1': 'data/velocity_models/monitoring_stage1.bin',
    'monitoring_stage2': 'data/velocity_models/monitoring_stage2.bin'
}

# Define the dimensions and interval (assuming they are the same for all files)
nx = 601  # Number of samples in the horizontal direction
nz = 501  # Number of samples in the vertical direction
interval = 2  # Spatial interval (e.g., meters)
expected_elements = nx * nz

loaded_data = {}

# Load and process each file
for name, file_path in file_info.items():
    print(f"\nAttempting to load and process: {file_path}")
    try:
        # Check if the file exists
        if not os.path.exists(file_path):
            print(f"Error: File not found at '{file_path}'. Skipping this file.")
            continue

        # Load the raw bytes
        # Use dtype=np.float32 or np.float64 depending on how the data was saved
        data_bytes = np.fromfile(file_path, dtype=np.float32) # Adjust dtype if necessary

        # Check if the number of elements matches the expected dimensions
        if data_bytes.size != expected_elements:
            print(f"Warning: Loaded data size ({data_bytes.size}) from '{file_path}' does not match expected size ({expected_elements}) based on dimensions nx={nx}, nz={nz}.")
            # Decide how to handle this: skip, try reshaping anyway, or raise error
            # For now, we'll attempt to reshape if it's possible, but warn the user.
            if data_bytes.size % (nx * nz) != 0:
                 print(f"Error: Data size ({data_bytes.size}) is not a multiple of expected dimensions ({nx}x{nz}). Cannot reshape.")
                 continue
            else:
                 print("Attempting to reshape anyway.")


        # Reshape the data to the specified dimensions
        # Assuming the data is stored row-by-row (C-order) and we want (nz, nx) shape
        try:
            data_2d = data_bytes.reshape((nx, nz)).T
        except ValueError:
            print(f"Error: Could not reshape data from '{file_path}' to ({nz}, {nx}). Check dimensions or dtype.")
            continue


        print(f"Successfully loaded and reshaped '{name}'. Shape: {data_2d.shape}")
        loaded_data[name] = data_2d

        # To visualize the data (optional, uncomment to enable)
        plt.figure(figsize=(12, 5))
        plt.imshow(data_2d, aspect='auto', cmap='seismic', extent=[0, nx*interval, nz*interval, 0]) # Use extent for physical dimensions
        plt.title(f'Loaded Binary Data: {name}')
        plt.xlabel('Horizontal Distance (m)')
        plt.ylabel('Depth/Time (m or samples)')
        plt.colorbar(label='Amplitude')
        plt.show()

    except FileNotFoundError:
        # This block is now less likely to be hit due to the os.path.exists check, but kept as a fallback
        print(f"Error: File not found at '{file_path}'. Skipping this file.")
    except Exception as e:
        print(f"An unexpected error occurred while processing '{file_path}': {e}")

# You can now access the loaded data using the dictionary 'loaded_data'
# For example:
if 'baseline' in loaded_data:
    baseline_data = loaded_data['baseline']
    print(f"\n'baseline_data' variable is available with shape {baseline_data.shape}")
if 'monitoring_stage1' in loaded_data:
    monitoring_stage1_data = loaded_data['monitoring_stage1']
    print(f"'monitoring_stage1_data' variable is available with shape {monitoring_stage1_data.shape}")
if 'monitoring_stage2' in loaded_data:
    monitoring_stage2_data = loaded_data['monitoring_stage2']
    print(f"'monitoring_stage2_data' variable is available with shape {monitoring_stage2_data.shape}")

# Example of accessing and using the data:
# if 'baseline' in loaded_data and 'monitoring_stage1' in loaded_data:
#     difference_data = loaded_data['monitoring_stage1'] - loaded_data['baseline']
#     print(f"\nCalculated difference between monitoring_stage1 and baseline. Shape: {difference_data.shape}")

## 2. Acoustic forward modeling with stochastic source scattering

Source coordinates are perturbed in the continuous domain *before* the wave equation is
solved, so the recorded wavefield carries the true distortion of physical mispositioning.

In [ ]:
import torch
import numpy as np
import deepwave
from deepwave import scalar
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from matplotlib.colors import TwoSlopeNorm

# Assuming 'baseline_data', 'monitoring_stage1_data', 'monitoring_stage2_data' are loaded
# Assuming 'device' is defined (e.g., torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

def run_deepwave_simulation(vp_model_data, n_shots, n_receivers_per_shot, max_offset_m, dx, dt, freq, nt, peak_time, source_depth, receiver_depth, first_source, first_receiver, model_name):
    """
    Runs a Deepwave scalar simulation with specified parameters and plots the geometry and gathers.

    Args:
        vp_model_data (np.ndarray): The Vp model data (NumPy array).
        n_shots (int): Number of shots.
        n_receivers_per_shot (int): Number of receivers per shot.
        max_offset_m (float): Maximum random source offset in meters.
        dx (float): Grid spacing in meters.
        dt (float): Time step in seconds.
        freq (float): Ricker wavelet dominant frequency.
        nt (int): Number of time steps.
        peak_time (float): Peak time of the Ricker wavelet.
        source_depth (int): Source depth in grid points.
        receiver_depth (int): Receiver depth in grid points.
        first_source (int): First source position in grid points.
        first_receiver (int): First receiver position in grid points.
        model_name (str): Name of the model (for printing/description and plot titles).

    Returns:
        torch.Tensor: The receiver amplitudes tensor.
    """
    print(f"\nRunning Deepwave simulation for: {model_name}")

    # Convert NumPy model to PyTorch tensor
    vp_model_tensor = torch.from_numpy(vp_model_data).to(device, dtype=torch.float32)
    ny, nx = vp_model_tensor.shape

    # Calculate source and receiver locations
    source_locations = torch.zeros(n_shots, 1, 2, dtype=torch.long, device=device)
    receiver_locations = torch.zeros(n_shots, n_receivers_per_shot, 2, dtype=torch.long, device=device)

    # Set Source Locations with Randomization
    max_offset_grid = int(max_offset_m / dx)
    regular_locations = torch.arange(n_shots, device=device, dtype=torch.float32) * (nx / n_shots) + first_source
    random_offsets = torch.randint(-max_offset_grid, max_offset_grid + 1, (n_shots,), device=device, dtype=torch.float32)
    randomized_x_locations = regular_locations + random_offsets
    randomized_x_locations.clamp_(0, nx - 1)
    source_locations[..., 1] = source_depth
    source_locations[:, 0, 0] = randomized_x_locations.long()

    print(f"Randomized source locations created with max spacing of +/- {max_offset_m} m.")

    # Set Receiver Locations (no randomization)
    receiver_locations[..., 1] = receiver_depth
    receiver_locations[:, :, 0] = (torch.arange(n_receivers_per_shot) * (nx / n_receivers_per_shot) + first_receiver).repeat(n_shots, 1)

    # Set Source Amplitudes
    source_amplitudes = (
        deepwave.wavelets.ricker(freq, nt, dt, peak_time)
        .repeat(n_shots, 1, 1)
        .to(device)
    )

    # =========================================
    # Plot Survey Geometry
    # =========================================
    print("\nPlotting survey geometry...")

    dz = dx # Assuming grid spacing is the same in x and z directions

    # Convert source/receiver locations from grid points to meters
    source_x_m = source_locations[..., 0].cpu().numpy() * dx
    source_z_m = source_locations[..., 1].cpu().numpy() * dz

    receiver_x_m = receiver_locations[..., 0].cpu().numpy() * dx
    receiver_z_m = receiver_locations[..., 1].cpu().numpy() * dz

    fig, ax = plt.subplots(figsize=(18, 8))

    # Marker settings
    source_size = 50
    receiver_size = 25
    source_marker = '*'
    receiver_marker = 'v'

    # Use 'extent' to set the correct axis labels in meters
    extent = [0, nx * dx, ny * dz, 0] # [left, right, bottom, top]
    im = ax.imshow(vp_model_tensor.cpu(), aspect='auto', cmap='jet',
                   extent=extent)

    # Plot the scaled source and receiver locations
    scatter_sources = ax.scatter(source_x_m.flatten(), source_z_m.flatten(), c='r', label='Sources', s=source_size, marker=source_marker)
    scatter_receivers = ax.scatter(receiver_x_m.flatten(), receiver_z_m.flatten(), c='b', label='Receivers', s=receiver_size, marker=receiver_marker)

    # Create custom legend handles (bigger symbols just for legend)
    source_patch = plt.Line2D([0], [0], marker=source_marker, color='w', label='Sources',
                              markerfacecolor='r', markersize=15, linestyle='None')
    receiver_patch = plt.Line2D([0], [0], marker=receiver_marker, color='w', label='Receivers',
                                markerfacecolor='b', markersize=10, linestyle='None')

    # Labels and formatting
    ax.set_title(f"Survey Geometry on Velocity Model\n({model_name})", fontsize=22)
    ax.set_xlabel("Horizontal Distance (m)", fontsize=16)
    ax.set_ylabel("Depth (m)", fontsize=16)
    ax.legend(handles=[source_patch, receiver_patch], fontsize=14)
    ax.tick_params(axis='both', labelsize=14)

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('P-wave Velocity (m/s)', fontsize=16)
    cbar.ax.tick_params(labelsize=14)

    # Save plot
    output_dir = 'outputs/figures'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    output_filename = os.path.join(output_dir, f'geometry_for_model_{model_name}.png')
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Plot saved to: {output_filename}")

    plt.show()


    # =========================================
    # DEEPWAVE FORWARD SIMULATION
    # =========================================

    out_sim = scalar(
        vp_model_tensor.T, dx, dt,
        source_amplitudes=source_amplitudes,
        source_locations=source_locations,
        receiver_locations=receiver_locations,
        accuracy=4,
        pml_freq=freq
    )

    receiver_amplitudes = out_sim[-1]
    print("Deepwave simulation complete.")
    print(f"{model_name} amplitudes tensor shape: {receiver_amplitudes.shape}")

    # =========================================
    # Plot Shot and Receiver Gathers
    # =========================================
    print("\nPlotting shot and receiver gathers...")

    # Get dimensions from the amplitudes tensor
    n_shots, n_receivers_per_shot, nt = receiver_amplitudes.shape

    # --- Dynamically select the middle shot and receiver ---
    middle_shot_idx = n_shots // 2
    middle_receiver_idx = n_receivers_per_shot // 2

    middle_shot_gather = receiver_amplitudes[middle_shot_idx, :, :]
    middle_receiver_gather = receiver_amplitudes[:, middle_receiver_idx, :]

    # Compute global min/max from both gathers for consistent color scaling
    all_data = torch.cat([middle_shot_gather.flatten(), middle_receiver_gather.flatten()])
    # Using a slightly smaller percentile can sometimes show more detail
    vmin, vmax = torch.quantile(all_data.cpu(), torch.tensor([0.02, 0.98])).tolist()

    vabs = max(abs(vmin), abs(vmax))
    vmin, vmax = -vabs, vabs

    # Create a diverging normalization centered at 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vabs) # Use vabs for symmetric range

    # --- Plotting ---
    fig, ax = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

    # --- Use 'extent' to create a physical time axis (in seconds) ---
    # Assuming dt is defined
    time_axis = nt * dt
    extent_shot = [0, n_receivers_per_shot, time_axis, 0]
    extent_receiver = [0, n_shots, time_axis, 0]

    im0 = ax[0].imshow(middle_shot_gather.cpu().T, aspect='auto', cmap='seismic', norm=norm, extent=extent_shot)
    im1 = ax[1].imshow(middle_receiver_gather.cpu().T, aspect='auto', cmap='seismic', norm=norm, extent=extent_receiver)

    # Labels and titles
    ax[0].set_xlabel("Receiver Channel", fontsize=12)
    ax[0].set_ylabel("Time (s)", fontsize=12) # Label now reflects the new axis
    ax[1].set_xlabel("Shot Number", fontsize=12)
    ax[0].set_title(f'Shot Gather (Shot {middle_shot_idx})', fontsize=14)
    ax[1].set_title(f'Receiver Gather (Receiver {middle_receiver_idx})', fontsize=14)

    fig.suptitle(f'Seismic Gathers for {model_name}', fontsize=16, y=0.96)
    plt.tight_layout(rect=[0, 0, 1, 0.94]) # Adjust layout for suptitle

    # Save figure
    output_dir = 'outputs/figures'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    plot_filename = os.path.join(output_dir, f'shot_and_receiver_gathers_{model_name}.png')
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"Plot saved to: {plot_filename}")

    plt.show()


    return receiver_amplitudes, source_locations # Return both

In [ ]:

# =========================================
# SIMULATION PARAMETERS
# =========================================
dx = 2
dt = 0.004
freq = 25
nt = 300
peak_time = 1.5 / freq
d_receiver = nx / 300

# Source and receiver parameters (assuming these are consistent across simulations)
source_depth_grid = int(60)
receiver_depth_grid = int(50)
first_source_grid = 5
first_receiver_grid = 0

# Define simulation configurations
sim_configs = {
    'baseline_good_repitability': {
        'data': baseline_data,
        'n_shots': 100,
        'n_receivers_per_shot': 300,
        'max_offset_m': 0.0,
        'name': 'baseline_good_repitability'
    },
    'monitoring_stage1_good_repitability': {
        'data': monitoring_stage1_data,
        'n_shots': 100,
        'n_receivers_per_shot': 300,
        'max_offset_m': 0.0,
        'name': 'monitoring_stage1_good_repitability'
    },
    'monitoring_stage2_good_repitability': {
        'data': monitoring_stage2_data,
        'n_shots': 100,
        'n_receivers_per_shot': 300,
        'max_offset_m': 0.0,
        'name': 'monitoring_stage2_good_repitability'
    },
    'baseline_bad_repitability': {
        'data': baseline_data,
        'n_shots': 10,
        'n_receivers_per_shot': 300,
        'max_offset_m': 130.0,
        'name': 'baseline_bad_repitability'
    },
    'monitoring_stage1_bad_repitability': {
        'data': monitoring_stage1_data,
        'n_shots': 10,
        'n_receivers_per_shot': 300,
        'max_offset_m': 130.0,
        'name': 'monitoring_stage1_bad_repitability'
    },
    'monitoring_stage2_bad_repitability': {
        'data': monitoring_stage2_data,
        'n_shots': 10,
        'n_receivers_per_shot': 300,
        'max_offset_m': 130.0,
        'name': 'monitoring_stage2_bad_repitability'
    }
}

# Run simulations and store results
amplitudes_results = {}
locations_results = {} # <-- New dictionary for locations

for key, config in sim_configs.items():
    # The function now returns two items
    amplitudes, locations = run_deepwave_simulation(
        config['data'],
        config['n_shots'],
        config['n_receivers_per_shot'],
        config['max_offset_m'],
        dx, dt, freq, nt, peak_time,
        source_depth_grid, receiver_depth_grid, first_source_grid, first_receiver_grid,
        config['name']
    )
    # Store each item in its own dictionary
    amplitudes_results[key] = amplitudes
    locations_results[key] = locations

# Access results from the amplitudes_results dictionary
baseline_good_amplitudes_tensor = amplitudes_results['baseline_good_repitability']
monitoring_good_stage1_amplitudes_tensor = amplitudes_results['monitoring_stage1_good_repitability']
monitoring_stage2_good_amplitudes_tensor = amplitudes_results['monitoring_stage2_good_repitability']
baseline_bad_amplitudes_tensor = amplitudes_results['baseline_bad_repitability']
monitoring_stage1_bad_amplitudes_tensor = amplitudes_results['monitoring_stage1_bad_repitability']
monitoring_stage2_bad_amplitudes_tensor = amplitudes_results['monitoring_stage2_bad_repitability']

print("\nAll simulations completed. Amplitudes and Source Locations are stored.")

## 3. Pre-processing: F-K filter, CMP sort, NMO, stack

In [ ]:
import torch
import matplotlib.pyplot as plt
import os
from typing import Optional

# =========================================
# HELPER FOR COSINE TAPER
# =========================================

def _create_taper(n_dim: int, taper_frac: float, device: torch.device) -> torch.Tensor:
    """Creates a 1D cosine taper (Tukey window)."""
    if taper_frac <= 0.0 or taper_frac > 0.5:
        return torch.ones(n_dim, device=device)

    taper_len = int(n_dim * taper_frac)
    if taper_len == 0:
        return torch.ones(n_dim, device=device)

    # Create a Hann window twice the length of the taper
    hann_win = torch.hann_window(taper_len * 2, periodic=False).to(device)

    full_taper = torch.ones(n_dim, device=device)
    full_taper[:taper_len] = hann_win[:taper_len]
    full_taper[-taper_len:] = hann_win[-taper_len:]

    return full_taper

def cosine_taper(data: torch.Tensor, taper_frac: float = 0.1) -> torch.Tensor:
    """Apply a 2D cosine taper (Tukey window) to reduce edge effects."""
    n_traces, n_samples = data.shape

    taper_x = _create_taper(n_traces, taper_frac, data.device)
    taper_t = _create_taper(n_samples, taper_frac, data.device)

    # Apply tapers by broadcasting
    return data * taper_x.unsqueeze(1) * taper_t.unsqueeze(0)

# =========================================
# F-K VELOCITY FILTERING
# =========================================
# This section filters out low-velocity noise like ground roll from shot gathers.

def smooth_fk_mask(freqs: torch.Tensor, kx: torch.Tensor, vmin: float, vbuffer: float = 300.0) -> torch.Tensor:
    """Create a smooth mask in the f-k domain to retain energy above a minimum velocity."""
    # Create frequency and wavenumber grids. 'ij' indexing matches FFT output shape.
    KX, FREQS = torch.meshgrid(kx, freqs, indexing='ij')

    # Calculate apparent velocity: V = f / k. Add epsilon to avoid division by zero.
    velocity = torch.abs(FREQS / (KX + 1e-10))

    # Create a smooth mask using a hyperbolic tangent (sigmoid-like) function
    mask = 0.5 * (1 + torch.tanh((velocity - vmin) / vbuffer))
    return mask

def fk_velocity_filter(data: torch.Tensor, dt: float, dx: float, vmin: float, vbuffer: float = 300.0) -> torch.Tensor:
    """Apply a velocity-based f-k filter with smooth masking and edge tapering."""
    if data.ndim != 2:
        raise ValueError(f"Input data for fk_velocity_filter must be 2D, but got {data.shape}")

    # 1. Apply taper to reduce FFT artifacts from data edges
    data_tapered = cosine_taper(data, taper_frac=0.05)

    # 2. 2D Fourier Transform to move from (x, t) to (k, f) domain
    data_fk = torch.fft.fft2(data_tapered)

    # 3. Generate frequency and wavenumber axes for the mask
    n_traces, n_samples = data_tapered.shape
    freqs = torch.fft.fftfreq(n_samples, d=dt).to(data.device)
    kx = torch.fft.fftfreq(n_traces, d=dx).to(data.device)

    # 4. Create the filter mask
    mask = smooth_fk_mask(freqs, kx, vmin=vmin, vbuffer=vbuffer)

    # 5. Apply the mask in the f-k domain
    data_fk_filtered = data_fk * mask

    # 6. Inverse 2D Fourier Transform to return to (x, t) domain
    filtered_data = torch.fft.ifft2(data_fk_filtered)

    return filtered_data.real

# =========================================
# REFACTORED APPLICATION FUNCTIONS
# =========================================

def apply_fk_to_batch(data_batch_tensor: torch.Tensor, dt: float, dx_physical: float,
                      vmin: float, vbuffer: float = 300.0) -> torch.Tensor:
    """
    Applies the 2D F-K filter to each shot gather in a 3D batch.

    Args:
        data_batch_tensor (torch.Tensor): The 3D input data (n_shots, n_traces, n_samples).
        dt (float): Time sampling interval (in seconds).
        dx_physical (float): Physical distance between traces (in meters).
        vmin (float): Minimum velocity to keep (m/s).
        vbuffer (float): Velocity transition buffer (m/s).

    Returns:
        torch.Tensor: The 3D tensor of filtered shot gathers.
    """
    n_shots = data_batch_tensor.shape[0]
    print(f"\nApplying F-K filter to {n_shots} shot gathers...")

    filtered_batch = torch.zeros_like(data_batch_tensor)

    # Loop over each shot gather and apply the filter
    for i in range(n_shots):
        shot_gather = data_batch_tensor[i]
        filtered_batch[i] = fk_velocity_filter(
            shot_gather,
            dt=dt,
            dx=dx_physical,
            vmin=vmin,
            vbuffer=vbuffer
        )

    print(f"F-K filtering complete. Filtered gathers shape: {filtered_batch.shape}")
    return filtered_batch

def plot_fk_comparison(original_data: torch.Tensor, filtered_data: torch.Tensor,
                       dataset_name: str, save_dir: str,
                       shot_idx: Optional[int] = None):
    """
    Generates and saves a side-by-side comparison plot of original vs. filtered data.

    Args:
        original_data (torch.Tensor): The 3D original data (n_shots, n_traces, n_samples).
        filtered_data (torch.Tensor): The 3D filtered data (n_shots, n_traces, n_samples).
        dataset_name (str): A name for the dataset (e.g., "baseline_good_repitability").
        save_dir (str): The directory to save the plot.
        shot_idx (int, optional): The specific shot index to plot.
                                  If None, plots the middle shot.
    """
    n_shots = original_data.shape[0]
    if shot_idx is None:
        shot_idx = n_shots // 2

    print(f"Generating comparison plot for {dataset_name}, shot {shot_idx}...")

    original_shot = original_data[shot_idx].cpu()
    filtered_shot = filtered_data[shot_idx].cpu()

    # Use 98th percentile of original data for a consistent color scale
    vabs = torch.quantile(torch.abs(original_shot), 0.98)

    fig, ax = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

    # Plot Original
    ax[0].imshow(original_shot.T, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax[0].set_title(f"Original: {dataset_name} (Shot {shot_idx})")
    ax[0].set_xlabel("Channel")
    ax[0].set_ylabel("Time Sample")

    # Plot Filtered
    ax[1].imshow(filtered_shot.T, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax[1].set_title(f"F-K Filtered: {dataset_name} (Shot {shot_idx})")
    ax[1].set_xlabel("Channel")

    plt.tight_layout()

    # Ensure the save directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Save the figure with a unique name
    fk_plot_filename = os.path.join(save_dir, f"fk_filter_comparison_{dataset_name}_shot_{shot_idx}.png")
    plt.savefig(fk_plot_filename, dpi=300, bbox_inches='tight')
    print(f"F-K filter comparison plot saved to: {fk_plot_filename}")
    plt.show()

# =========================================
# MAIN APPLICATION AND VISUALIZATION
# =========================================

# --- 1. Set up processing parameters ---
receiver_spacing_m = d_receiver * dx
vmin_filter = 1900.0  # Min velocity to keep (e.g., water velocity)
vbuffer_filter = 300.0
save_directory = 'outputs/figures/'

# --- 2. Create a dictionary of datasets to process ---
# This makes it easy to add or remove datasets
datasets_to_process = {
    "baseline_good_repitability": baseline_good_amplitudes_tensor,
    "monitoring_stage1_good_repitability": monitoring_good_stage1_amplitudes_tensor,
    "monitoring_stage2_good_repitability": monitoring_stage2_good_amplitudes_tensor,
    "baseline_bad_repitability": baseline_bad_amplitudes_tensor,
    "monitoring_stage1_bad_repitability": monitoring_stage1_bad_amplitudes_tensor,
    "monitoring_stage2_bad_repitability": monitoring_stage2_bad_amplitudes_tensor,
}

# This dictionary will hold all the filtered results
filtered_datasets = {}

# --- 3. Run the main processing loop ---
for name, data_tensor in datasets_to_process.items():

    # Apply the F-K filter to the entire batch of shots
    filtered_data = apply_fk_to_batch(
        data_tensor,
        dt=dt,
        dx_physical=receiver_spacing_m,
        vmin=vmin_filter,
        vbuffer=vbuffer_filter
    )

    # Store the result with a new name
    # e.g., "baseline_good_repitability" -> "baseline_good_repitability_filtered"
    filtered_datasets[f"{name}_filtered"] = filtered_data

    # Plot the comparison for the middle shot
    plot_fk_comparison(
        original_data=data_tensor,
        filtered_data=filtered_data,
        dataset_name=name,
        save_dir=save_directory
        # `shot_idx` is omitted, so it will default to the middle shot
    )

print("\n--- All processing complete. ---")
print("Filtered datasets available in the 'filtered_datasets' dictionary:")
print(list(filtered_datasets.keys()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.ndimage import gaussian_filter1d
import os

# =========================================
# FULL PROCESSING PIPELINE (AS A FUNCTION)
# =========================================
def process_and_stack_dataset(seismic_data: torch.Tensor,
                              dataset_name: str,
                              source_locations_for_this_data: torch.Tensor,
                              save_dir_figures: str,
                              save_dir_numpy: str):
    """
    Runs the full CMP sort, NMO, stack, and smooth pipeline for a given dataset.
    Returns the final stacked numpy array.
    """

    print(f"\n=======================================================")
    print(f"Processing dataset: {dataset_name}")
    print(f"=======================================================")

    # =========================================
    # CMP SORTING AND GATHERING
    # =========================================
    print("Sorting data into CMP gathers...")
    num_shots, num_receivers, num_samples = seismic_data.shape
    source_positions = source_locations_for_this_data[:, 0, 0].float() * dx
    receiver_spacing_m = d_receiver * dx
    receiver_positions = (torch.arange(num_receivers, device=device) * receiver_spacing_m) + (first_receiver_grid * dx)
    all_cmps = (source_positions.view(-1, 1) + receiver_positions.view(1, -1)) / 2
    all_offsets = torch.abs(source_positions.view(-1, 1) - receiver_positions.view(1, -1))

    print("Defining a fixed CMP grid...")
    fixed_cmp_min_m = 0.0
    fixed_cmp_max_m = nx * dx
    cmp_bin_size = receiver_spacing_m / 2
    global cmp_bins # Make cmp_bins global so the plotting function can see it
    cmp_bins = torch.arange(fixed_cmp_min_m, fixed_cmp_max_m, cmp_bin_size, device=device)
    num_cmp_bins = len(cmp_bins)
    print(f"Fixed CMP grid created with {num_cmp_bins} bins.")

    bin_indices = torch.bucketize(all_cmps, cmp_bins)
    cmp_gathers_list = [None] * num_cmp_bins
    unique_bins, inverse_indices = torch.unique(bin_indices, return_inverse=True)
    for i, bin_idx_val in enumerate(unique_bins):
        bin_idx = bin_idx_val.item()
        if 0 <= bin_idx < num_cmp_bins:
            mask = (bin_indices == bin_idx_val)
            traces = seismic_data[mask]
            offsets = all_offsets[mask]
            sorted_idx = torch.argsort(offsets)
            cmp_gathers_list[bin_idx] = {'traces': traces[sorted_idx], 'offsets': offsets[sorted_idx]}
    print(f"CMP gathering complete. Found {len([g for g in cmp_gathers_list if g is not None])} non-empty CMPs.")

    # =========================================
    # NMO CORRECTION AND STACKING
    # =========================================
    print("\nApplying NMO correction with spatially-varying velocity, muting, and stacking...")

    def apply_nmo_correction(gather_data, dt, v_nmo_func, stretch_max=0.5):
        traces = gather_data['traces']
        offsets = gather_data['offsets']
        n_traces, n_samples = traces.shape
        if n_traces == 0: return None

        t0 = (torch.arange(n_samples, device=device, dtype=torch.float32) * dt).view(1, -1)
        x = offsets.view(-1, 1)

        if v_nmo_func.ndim == 1:
            v_nmo_func = v_nmo_func.view(1, -1)

        t_nmo = torch.sqrt(t0**2 + (x / (v_nmo_func + 1e-9))**2)

        # --- NMO Stretch Calculation & Mute ---
        stretch = (t_nmo - t0) / (t0 + 1e-6) # Added 1e-6 to avoid div by zero

        # Create a smooth taper mask to mute severely stretched far-offset traces
        stretch_taper_start = stretch_max - 0.15
        stretch_weights = 1.0 - torch.clamp((stretch - stretch_taper_start) / 0.15, min=0.0, max=1.0)
        # --------------------------------------

        sample_nmo = t_nmo / dt
        sample_floor = torch.floor(sample_nmo).long().clamp(0, n_samples - 2)
        sample_ceil = sample_floor + 1
        weight_ceil = sample_nmo - sample_floor.float()

        val_floor = torch.gather(traces, 1, sample_floor)
        val_ceil = torch.gather(traces, 1, sample_ceil)

        corrected_traces = val_floor * (1.0 - weight_ceil) + val_ceil * weight_ceil

        # Apply the stretch mute weight mask to the corrected traces
        corrected_traces = corrected_traces * stretch_weights

        return corrected_traces

    def apply_top_mute(corrected_gather, offsets, dt, slope=0.5, t0=0.1, taper_length_s=0.08):
        n_traces, n_samples = corrected_gather.shape
        mute_times = t0 + offsets * (slope / 1000)
        mute_samples = (mute_times / dt)
        time_samples = torch.arange(n_samples, device=device).expand_as(corrected_gather)

        # --- Smooth Tapered Top Mute ---
        dist_from_mute = time_samples - mute_samples.unsqueeze(1)
        taper_samples = taper_length_s / dt
        weights = torch.clamp(dist_from_mute / taper_samples, min=0.0, max=1.0)
        # -------------------------------

        return corrected_gather * weights

    print("Building 2D NMO velocity field...")
    nmo_velocities = torch.zeros(num_cmp_bins, num_samples, device=device)
    time_axis = torch.arange(num_samples, device=device) * dt
    # Use baseline_data (from sim cell) to build the NMO field
    vp_model_tensor = torch.from_numpy(baseline_data).to(device, dtype=torch.float32)

    for i in range(num_cmp_bins):
        cmp_location_m = cmp_bins[i]
        x_idx = int(round((cmp_location_m / dx).item()))
        x_idx = min(x_idx, vp_model_tensor.shape[1] - 1)
        v_profile_depth = vp_model_tensor[:, x_idx]
        dz = dx
        twt_profile = torch.zeros_like(v_profile_depth, dtype=torch.float32)
        twt_profile[1:] = torch.cumsum(2 * dz / v_profile_depth[:-1], dim=0)
        v_nmo_t = torch.from_numpy(
            np.interp(time_axis.cpu().numpy(), twt_profile.cpu().numpy(), v_profile_depth.cpu().numpy())
        ).to(device)
        nmo_velocities[i, :] = v_nmo_t

    stacked_section = torch.zeros(num_cmp_bins, num_samples, device=device)
    nmo_corrected_example = None
    original_cmp_example = None
    example_idx = num_cmp_bins // 2

    for i, gather in enumerate(cmp_gathers_list):
        if gather is not None and len(gather['offsets']) > 0:
            velocity_for_this_cmp = nmo_velocities[i]
            corrected_traces = apply_nmo_correction(gather, dt, velocity_for_this_cmp)
            if corrected_traces is not None:
                muted_traces = apply_top_mute(corrected_traces, gather['offsets'], dt)
                stacked_section[i] = torch.mean(muted_traces, dim=0)
                if i == example_idx:
                    nmo_corrected_example = muted_traces
                    original_cmp_example = gather['traces']
    print("NMO, muting, and stacking complete.")

    # =========================================
    # VISUALIZATION of NMO
    # =========================================
    if original_cmp_example is not None:
        fig, ax = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
        vabs_cmp = torch.quantile(torch.abs(original_cmp_example.cpu()), 0.98)
        ax[0].imshow(original_cmp_example.cpu().T, cmap='seismic', aspect='auto', vmin=-vabs_cmp, vmax=vabs_cmp)
        ax[0].set_title(f'CMP Gather #{example_idx} (Before NMO)\n{dataset_name}')
        ax[0].set_xlabel('Trace Number (sorted by offset)')
        ax[0].set_ylabel('Time Sample')
        ax[1].imshow(nmo_corrected_example.cpu().T, cmap='seismic', aspect='auto', vmin=-vabs_cmp, vmax=vabs_cmp)
        ax[1].set_title(f'CMP Gather #{example_idx} (After NMO & Mute)\n{dataset_name}')
        ax[1].set_xlabel('Trace Number (sorted by offset)')
        plt.tight_layout()
        os.makedirs(save_dir_figures, exist_ok=True)
        nmo_plot_filename = os.path.join(save_dir_figures, f'nmo_comparison_gather_{example_idx}_({dataset_name}).png')
        plt.savefig(nmo_plot_filename, dpi=300, bbox_inches='tight')
        print(f"NMO comparison plot saved to: {nmo_plot_filename}")
        plt.show()

    # =========================================
    # APPLY LATERAL SMOOTHING
    # =========================================
    print("Applying lateral smoothing...")
    stacked_np = stacked_section.cpu().numpy()
    sigma_lateral = 1.0
    stacked_smoothed = gaussian_filter1d(stacked_np, sigma=sigma_lateral, axis=0)

    # =========================================
    # SAVE THE FINAL CMP SECTION
    # =========================================
    os.makedirs(save_dir_numpy, exist_ok=True)
    output_filename = os.path.join(save_dir_numpy, f'final_stacked_cmp_section_({dataset_name}).npy')
    np.save(output_filename, stacked_smoothed)
    print(f"\n✅ Final CMP section successfully saved to: {output_filename}  with shape {stacked_smoothed.shape}")

    # =========================================
    # PLOT THE SMOOTHED STACKED SECTION
    # =========================================
    plt.figure(figsize=(14, 9))
    valid_data = stacked_smoothed[np.isfinite(stacked_smoothed)]
    if valid_data.size > 0:
        vabs_stack = np.percentile(np.abs(valid_data), 99)
    else:
        vabs_stack = 1.0

    # Use global cmp_bins, nt, dt for plot extent
    plt.imshow(stacked_smoothed.T, cmap='seismic', aspect='auto',
               vmin=-vabs_stack, vmax=vabs_stack,
               extent=[cmp_bins[0].item(), cmp_bins[-1].item(), nt*dt, 0])
    plt.title(f'Final Stacked Section (Smoothed)\n{dataset_name}', fontsize=16)
    plt.xlabel('CMP Position (m)', fontsize=12)
    plt.ylabel('Two-Way Time (s)', fontsize=12)
    plt.colorbar(label='Amplitude')

    stack_plot_filename = os.path.join(save_dir_figures, f'final_stacked_section_({dataset_name}).png')
    plt.savefig(stack_plot_filename, dpi=300, bbox_inches='tight')
    print(f"Final stacked section plot saved to: {stack_plot_filename}")
    plt.show()

    # Return the final result
    return stacked_smoothed


# =========================================
# MAIN EXECUTION LOOP (FIXED)
# =========================================
# Define save locations
figures_directory = 'outputs/figures'
numpy_directory = 'outputs/stacked'

# This dictionary will hold all the final stacked .npy data
final_stacked_datasets = {} # <-- Initialize as an empty dictionary

print("\n--- Starting full processing pipeline for all datasets ---")

# Loop through all the F-K filtered datasets
for filtered_name, data_tensor in filtered_datasets.items():

    # Get the base name, e.g., "baseline_good_repitability"
    base_name = filtered_name.replace("_filtered", "")

    # Get the correct source locations for this base_name
    locations_for_this_dataset = locations_results[base_name]

    # 1. Call the function and get the returned numpy array
    stacked_result = process_and_stack_dataset(
        seismic_data=data_tensor,
        dataset_name=base_name,
        source_locations_for_this_data=locations_for_this_dataset,
        save_dir_figures=figures_directory,
        save_dir_numpy=numpy_directory
    )

    # 2. Assign that array to a key in the dictionary
    final_stacked_datasets[base_name] = stacked_result

print("\n--- All datasets have been processed and stored in 'final_stacked_datasets' dictionary. ---")
print(f"Available keys: {list(final_stacked_datasets.keys())}")


# ==========================================================
# TIME-LAPSE CALCULATION AND PLOTTING
# ==========================================================
print("\n--- Starting Time-Lapse Difference Calculation ---")

# --- 1. Get data directly from the dictionary ---
print("Accessing stacked data from in-memory dictionary...")
stack_base_good = final_stacked_datasets['baseline_good_repitability']
stack_mon1_good = final_stacked_datasets['monitoring_stage1_good_repitability']
stack_mon2_good = final_stacked_datasets['monitoring_stage2_good_repitability']

stack_base_bad = final_stacked_datasets['baseline_bad_repitability']
stack_mon1_bad = final_stacked_datasets['monitoring_stage1_bad_repitability']
stack_mon2_bad = final_stacked_datasets['monitoring_stage2_bad_repitability']

# --- 2. Calculate Differences ---
print("Calculating time-lapse differences...")
diff_good_s1_base = stack_mon1_good - stack_base_good
diff_good_s2_base = stack_mon2_good - stack_base_good

diff_bad_s1_base = stack_mon1_bad - stack_base_bad
diff_bad_s2_base = stack_mon2_bad - stack_base_bad

# --- 3. Define Plotting Helper Function ---
try:
    plot_extent = [cmp_bins[0].item(), cmp_bins[-1].item(), nt*dt, 0] # [left, right, bottom, top]
except NameError as e:
    print(f"Warning: Could not get plot extent from global variables ({e}). Using default.")
    plot_extent = [0, 1, 1, 0] # Fallback

def plot_time_lapse(data_to_plot, title_str, save_filename_str):
    """
    Helper function to plot a single time-lapse difference section.
    """
    plt.figure(figsize=(14, 9))

    # Use a symmetric color scale centered at zero
    valid_data = data_to_plot[np.isfinite(data_to_plot)]
    if valid_data.size > 0:
        vabs_diff = np.percentile(np.abs(valid_data), 98)
    else:
        vabs_diff = 1.0

    if vabs_diff == 0: # Avoid vmin=0, vmax=0
        vabs_diff = 1.0

    plt.imshow(data_to_plot.T, cmap='seismic', aspect='auto',
               vmin=-vabs_diff, vmax=vabs_diff,
               extent=plot_extent)

    plt.title(title_str, fontsize=16)
    plt.xlabel('CMP Position (m)', fontsize=12)
    plt.ylabel('Two-Way Time (s)', fontsize=12)
    plt.colorbar(label='Amplitude Difference')

    os.makedirs(figures_directory, exist_ok=True)
    full_save_path = os.path.join(figures_directory, save_filename_str)
    plt.savefig(full_save_path, dpi=300, bbox_inches='tight')
    print(f"Time-lapse plot saved to: {full_save_path}")
    plt.show()

# --- 4. Generate All 4 Plots ---
print("\nGenerating and saving time-lapse plots...")

# Good Repeatability Plots
plot_time_lapse(
    diff_good_s1_base,
    'Time-Lapse (Good Rep): Stage 1 - Baseline',
    'timelapse_diff_good_stage1_vs_baseline.png'
)

plot_time_lapse(
    diff_good_s2_base,
    'Time-Lapse (Good Rep): Stage 2 - Baseline',
    'timelapse_diff_good_stage2_vs_baseline.png'
)

# Bad Repeatability Plots
plot_time_lapse(
    diff_bad_s1_base,
    'Time-Lapse (Bad Rep): Stage 1 - Baseline',
    'timelapse_diff_bad_stage1_vs_baseline.png'
)

plot_time_lapse(
    diff_bad_s2_base,
    'Time-Lapse (Bad Rep): Stage 2 - Baseline',
    'timelapse_diff_bad_stage2_vs_baseline.png'
)

print("\n--- Time-lapse plotting complete. ---")

## 4. Patch extraction, augmentation and splits

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import Model
from PIL import Image as PILImage, ImageDraw, ImageFont
import keras_tuner as kt


In [ ]:
import tensorflow as tf

# Keras layers can be used as functions for augmentation
# These are GPU-accelerated and highly efficient
random_flipper = tf.keras.layers.RandomFlip("horizontal")
random_rotator = tf.keras.layers.RandomRotation(factor=0.03) # Small rotation for "tilt"
random_translator = tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05) # For "stretch"

def augment(x, y):
    """
    Applies a random set of augmentations to a pair of low-res (x) and high-res (y) patches.
    """
    # --- 1. Apply Geometric Augmentations ---
    # These must be applied to both x and y consistently.
    # We stack them so the random transformation is generated once and applied to both.
    images = tf.concat([x, y], axis=-1) # Temporarily stack along the channel axis

    # Apply a random flip (50% chance)
    images = random_flipper(images)

    # Apply a random small rotation
    images = random_rotator(images)

    # Apply a random small translation (stretch/shift)
    images = random_translator(images)

    # Unstack the images back into x and y
    x, y = tf.split(images, num_or_size_splits=2, axis=-1)

    # --- 2. Apply Noise Augmentation ---
    # This should ONLY be applied to the input (x) to make the model more robust.
    # Do not add noise to the target (y).
    if tf.random.uniform(()) > 0.5: # 50% chance of adding noise
        noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.05, dtype=x.dtype)
        x = x + noise

    return x, y

def normalize_data(data, method='minmax_symmetric'):
    """Normalizes a 2D seismic section."""
    if method == 'minmax_symmetric':  # Scale to [-1, 1] based on max absolute value
        max_abs = np.max(np.abs(data))
        if max_abs < 1e-9:  # Avoid division by zero
            return np.zeros_like(data)
        return data / max_abs

    elif method == 'minmax_01':  # Scale to [0, 1]
        min_val = np.min(data)
        max_val = np.max(data)
        range_val = max_val - min_val
        if range_val == 0:
            return np.zeros_like(data)
        return (data - min_val) / range_val

    elif method == 'per_trace_max_abs':  # Normalize each trace separately to [-1, 1]
        norm_data = np.zeros_like(data)
        for i in range(data.shape[0]):
            max_abs = np.max(np.abs(data[i, :]))
            if max_abs > 1e-9:
                norm_data[i, :] = data[i, :] / max_abs
        return norm_data

    else:  # No normalization
        return data


# --- 2. Data Loading and Patching ---
def sliding_window_view(arr, window_shape, step_size):
    # This function is correct as is.
    window_h, window_w = window_shape if isinstance(window_shape, tuple) else (window_shape, window_shape)
    step_y, step_x = step_size if isinstance(step_size, tuple) else (step_size, step_size)
    arr_h, arr_w = arr.shape
    out_h = (arr_h - window_h) // step_y + 1
    out_w = (arr_w - window_w) // step_x + 1
    new_shape = (out_h, out_w, window_h, window_w)
    stride_y, stride_x = arr.strides
    new_strides = (stride_y * step_y, stride_x * step_x, stride_y, stride_x)
    return np.lib.stride_tricks.as_strided(arr, shape=new_shape, strides=new_strides)

In [ ]:
print("Accessing stacked data from in-memory dictionary...")
stack_base_good = final_stacked_datasets['baseline_good_repitability']
stack_mon1_good = final_stacked_datasets['monitoring_stage1_good_repitability']
stack_mon2_good = final_stacked_datasets['monitoring_stage2_good_repitability']

stack_base_bad = final_stacked_datasets['baseline_bad_repitability']
stack_mon1_bad = final_stacked_datasets['monitoring_stage1_bad_repitability']
stack_mon2_bad = final_stacked_datasets['monitoring_stage2_bad_repitability']

In [ ]:
# --- 1. Parameters ---
WINDOW_SIZE = 128
STEP_SIZE = 5
TEST_SIZE_TEMP = 0.3
VALIDATION_SIZE_FINAL = 0.3
BATCH_SIZE = 32
RANDOM_STATE = 42

# --- 2. Data Loading, Normalizing, and Patching ---
# Group the arrays to process them efficiently in a loop
bad_stacks = [stack_base_bad, stack_mon1_bad, stack_mon2_bad]
good_stacks = [stack_base_good, stack_mon1_good, stack_mon2_good]

X_segments_list = []
Y_segments_list = []

print("Processing and patching datasets...")
for bad, good in zip(bad_stacks, good_stacks):
    # Transpose and normalize
    x_data = normalize_data(bad.T)
    y_data = normalize_data(good.T)

    # Apply sliding window view
    x_view = sliding_window_view(x_data, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
    y_view = sliding_window_view(y_data, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)

    # Reshape and add channel dimension
    x_seg = x_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE)[..., np.newaxis]
    y_seg = y_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE)[..., np.newaxis]

    X_segments_list.append(x_seg)
    Y_segments_list.append(y_seg)

# Concatenate all datasets together
X_segments = np.concatenate(X_segments_list, axis=0)
Y_segments = np.concatenate(Y_segments_list, axis=0)

# Process Z_segments (baseline_good) in case you still reference it later
Z_data = normalize_data(stack_base_good.T)
Z_view = sliding_window_view(Z_data, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
Z_segments = Z_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE)[..., np.newaxis]

print(f"Total independent X segments (Bad):  {len(X_segments)}")
print(f"Total independent Y segments (Good): {len(Y_segments)}")
print(f"Total independent Z segments (Base): {len(Z_segments)}")

# --- 3. Data Splitting ---
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X_segments, Y_segments, test_size=TEST_SIZE_TEMP, random_state=RANDOM_STATE
)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=VALIDATION_SIZE_FINAL, random_state=RANDOM_STATE
)
print(f"Split: {len(X_train)} train / {len(X_val)} val / {len(X_test)} test.")

# --- 4. TensorFlow Datasets ---
train_dataset = (tf.data.Dataset.from_tensor_slices((X_train, Y_train))
                 .cache()
                 .shuffle(1000)
                 .batch(BATCH_SIZE)
                 .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
                 .prefetch(tf.data.AUTOTUNE))

val_dataset = (tf.data.Dataset.from_tensor_slices((X_val, Y_val))
               .batch(BATCH_SIZE)
               .cache()
               .prefetch(tf.data.AUTOTUNE))

test_dataset = (tf.data.Dataset.from_tensor_slices((X_test, Y_test))
                .batch(BATCH_SIZE)
                .cache()
                .prefetch(tf.data.AUTOTUNE))


## 5. U-Net training

Architecture fixed at 32 base filters (see notebook 03); learning rate tuned by Hyperband.
Skip this and run section 6 to load the released weights instead.

In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

# --- 3. Hyperparameter Tuning Setup ---

def build_unet_tuner(hp):
    """Builds a U-Net model with 32 filters (fixed) and a tunable learning rate."""
    inputs = layers.Input(shape=(WINDOW_SIZE, WINDOW_SIZE, 1))

    # FIXED (was tunable): adopted from the capacity ablation --
    # 32 filters beat 128 decisively, and 64/128 both showed real training instability.
    f = 32

    # Helper function to keep architecture code clean
    def block(x, n):
        x = layers.Conv2D(n, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(n, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        return x

    # Encoding path
    c1 = block(inputs, f);   p1 = layers.MaxPooling2D()(c1)
    c2 = block(p1, f*2);     p2 = layers.MaxPooling2D()(c2)
    c3 = block(p2, f*4);     p3 = layers.MaxPooling2D()(c3)
    c4 = block(p3, f*8);     p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    c5 = block(p4, f*16)

    # Decoding path
    u6 = layers.Conv2D(f*8, 3, padding='same', activation='relu')(layers.UpSampling2D()(c5))
    c6 = block(layers.Concatenate()([u6, c4]), f*8)

    u7 = layers.Conv2D(f*4, 3, padding='same', activation='relu')(layers.UpSampling2D()(c6))
    c7 = block(layers.Concatenate()([u7, c3]), f*4)

    u8 = layers.Conv2D(f*2, 3, padding='same', activation='relu')(layers.UpSampling2D()(c7))
    c8 = block(layers.Concatenate()([u8, c2]), f*2)

    u9 = layers.Conv2D(f, 3, padding='same', activation='relu')(layers.UpSampling2D()(c8))
    c9 = block(layers.Concatenate()([u9, c1]), f)

    outputs = layers.Conv2D(1, 1, activation='linear')(c9)
    model = models.Model(inputs, outputs)

    # HYPERPARAMETER (still tuned): learning rate.
    hp_learning_rate = hp.Choice('learning_rate', values=[5e-3, 1e-3, 5e-4, 1e-4])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate, clipvalue=1.0),
                  loss=tf.keras.losses.MeanAbsoluteError(), metrics=['mae'])
    return model

# Instantiate the tuner
tuner = kt.Hyperband(
    build_unet_tuner,
    objective='val_loss',
    max_epochs=30,
    factor=3,
    directory='keras_tuner_dir',
    project_name='unet_seismic_tuning_32filters_new',
)

# --- 4. Run the Hyperparameter Search ---
print("\nStarting hyperparameter search (learning rate only, 32 filters fixed)... 🤖")

# Clear session callback to prevent memory leaks during tuning
class ClearMemory(tf.keras.callbacks.Callback):
    def on_trial_end(self, trial):
        tf.keras.backend.clear_session()

stop_early = EarlyStopping(monitor='val_loss', patience=5)
tuner.search(train_dataset, epochs=30, validation_data=val_dataset, callbacks=[stop_early, ClearMemory()])

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Search complete! 🏆
Filters (fixed): 32
Optimal learning rate: {best_hps.get('learning_rate')}
""")

# --- 5. Train the Best Model ---
print("\nTraining the best model with optimal hyperparameters...")

# SSIM metric -- monitor what we're actually evaluated on, instead of duplicating MAE.
# (Previously loss=MAE and metrics=['mae'] computed the identical value twice.)
def ssim_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=2.0))
    # max_val=2.0 because normalize_data() maps to roughly [-1, 1]

best_model = tuner.hypermodel.build(best_hps)
best_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_hps.get('learning_rate'), clipvalue=1.0),
    loss=tf.keras.losses.MeanAbsoluteError(),
    metrics=[ssim_metric]
)
best_model.summary()

# Callbacks
final_early_stop = EarlyStopping(
    monitor='val_loss', patience=15, verbose=1, restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
model_checkpoint = ModelCheckpoint(
    filepath='best_tuned_unet_32filters.keras', monitor='val_loss',
    save_best_only=True, mode='min', verbose=1
)

history = best_model.fit(
    train_dataset,
    epochs=3,
    validation_data=val_dataset,
    callbacks=[final_early_stop, reduce_lr, model_checkpoint]
)
print("🚀 Final model training complete.")

# --- 6. Plot Training History of the Best Model ---
print("\nPlotting final model training history...")
best_epoch = int(np.argmin(history.history['val_loss']))

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='Training Loss (MAE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MAE)')
plt.axvline(best_epoch, color='gray', linestyle=':', label=f'Best epoch ({best_epoch})')
plt.title('Loss (MAE)'); plt.xlabel('Epoch'); plt.ylabel('MAE')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(history.history['ssim_metric'], label='Training SSIM')
plt.plot(history.history['val_ssim_metric'], label='Validation SSIM')
plt.axvline(best_epoch, color='gray', linestyle=':')
plt.title('SSIM (higher is better)'); plt.xlabel('Epoch'); plt.ylabel('SSIM')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(history.history['learning_rate'] if 'learning_rate' in history.history else history.history.get('lr', []),
         label='Learning rate')
plt.title('Learning rate schedule'); plt.xlabel('Epoch'); plt.ylabel('LR')
plt.yscale('log'); plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best epoch {best_epoch}: "
      f"val MAE={history.history['val_loss'][best_epoch]:.4f}, "
      f"val SSIM={history.history['val_ssim_metric'][best_epoch]:.4f}")

## 6. Load the released trained model

In [ ]:
from tensorflow.keras.models import load_model
import tensorflow as tf

# Define the custom ssim_metric function as it was defined during model compilation
def ssim_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=2.0))

# Load the model, providing the custom_objects dictionary
best_model = load_model('models/unet_32filters.keras', custom_objects={'ssim_metric': ssim_metric})

## 7. Evaluation

In [ ]:
# --- 10. Evaluate on Test Set and Plot Results ---
print("\nEvaluating model on the test set...")
# Load the best model saved by ModelCheckpoint for evaluation

test_loss, test_mae = best_model.evaluate(X_test, Y_test, verbose=0)
print(f"Test Set Loss (MSE): {test_loss:.6f}")
print(f"Test Set Mean Absolute Error (MAE): {test_mae:.6f}")

print("\nPlotting predictions on a few random test samples...")
predicted_test_segments = best_model.predict(X_test)

num_samples_to_plot = 3
if len(X_test) >= num_samples_to_plot:
    random_indices = np.random.choice(len(X_test), num_samples_to_plot, replace=False)

    plt.figure(figsize=(6 * num_samples_to_plot, 9)) # Adjusted figsize
    for i, idx in enumerate(random_indices):
        # Low-Res Input (X_test)
        plt.subplot(3, num_samples_to_plot, i + 1)
        plt.imshow(X_test[idx].squeeze(), cmap="seismic") # Squeeze channel dim
        plt.title(f"Low-Res Input {idx}")
        plt.axis('off')

        # High-Res Ground Truth (Y_test)
        plt.subplot(3, num_samples_to_plot, i + 1 + num_samples_to_plot)
        plt.imshow(Y_test[idx].squeeze(), cmap="seismic")
        plt.title(f"High-Res Ground Truth {idx}")
        plt.axis('off')

        # U-Net Prediction
        plt.subplot(3, num_samples_to_plot, i + 1 + 2 * num_samples_to_plot)
        plt.imshow(predicted_test_segments[idx].squeeze(), cmap="seismic")
        plt.title(f"U-Net Prediction {idx}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Not enough test samples to plot the requested number.")


#predicted_test_segments = best_model.predict(X_segments)




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- 2. Perform Predictions & Reconstruct Full Images ---
print("Predicting on all segments...")
#predicted_segments = best_model.predict(X_segments, verbose=0)
STEP_SIZE=5
print("Reconstructing full images...")
recon_bad = reconstruct_from_patches_average(X_segments, X_data.shape, WINDOW_SIZE, STEP_SIZE)
recon_pred = reconstruct_from_patches_average(predicted_segments, X_data.shape, WINDOW_SIZE, STEP_SIZE)
recon_good = reconstruct_from_patches_average(Y_segments, X_data.shape, WINDOW_SIZE, STEP_SIZE)

# --- FIX 1: Remove Neural Network DC Bias ---
# U-Nets often output a tiny constant offset. If the average background is slightly negative,
# the 'seismic' colormap shifts from white to light blue. Subtracting the mean centers it back to 0.
recon_pred = recon_pred - np.mean(recon_pred)

# Calculate consistent color scale
# --- FIX 2: Use a data-driven symmetric scale instead of hardcoded -0.1 to 0.1 ---
vabs = np.percentile(np.abs(recon_good), 95) # 98th percentile prevents outliers from washing out the plot
vmin, vmax = -vabs, vabs

# --- 3. Pick 6 Random Segments ---
num_samples = 6
h, w = X_data.shape
random_origins = [(114, 125), (310, 170), (205, 785),
                  (430, 300), (10, 400), (385, 795)]

# --- 4. Plotting Setup ---
fig = plt.figure(figsize=(36, 18), constrained_layout=True)

# NEW: Changed to 4 rows. The `0.4` is the invisible gap between the top and the patches.
# Increase or decrease `0.4` to make the gap larger or smaller.
gs = fig.add_gridspec(4, 3, height_ratios=[2, 0.2, 1, 1])

colors = ['lime', 'red', 'yellow',
          'cyan', 'olive', 'black']

# --- ROW 1: Full Sections ---
titles = ['Input (Poor Reliability)',
          'U-Net Prediction',
          'Target (Good Reliability)']
data_full = [recon_bad, recon_pred, recon_good]

axs_top = []

for j in range(3):
    # This stays at row 0
    ax = fig.add_subplot(gs[0, j])

    # --- FIX 3: Removed the buggy 'if j == 9:' block ---
    # Since 'j' only goes from 0 to 2, j==9 was never reached.
    # We now use the consistent vmin/vmax for all plots so the shared colorbar is perfectly accurate.
    im = ax.imshow(data_full[j], cmap='seismic', aspect='auto', vmin=vmin, vmax=vmax)

    ax.set_title(titles[j], fontsize=26, pad=10, fontweight='bold')
    ax.set_xlabel('Trace Number', fontsize=25)
    if j == 0:
        ax.set_ylabel('Time Sample', fontsize=25)

    ax.tick_params(axis='both', which='major', labelsize=18)

    # Draw all 6 bounding boxes
    for i, (y, x) in enumerate(random_origins):
        rect = patches.Rectangle((x, y), WINDOW_SIZE, WINDOW_SIZE,
                                 linewidth=3, edgecolor=colors[i], facecolor='none', linestyle='--')
        ax.add_patch(rect)

    axs_top.append(ax)

# Add a colorbar safely attached to the collected axes
cbar = fig.colorbar(im, ax=axs_top, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Model Output Amplitude', fontsize=22)
cbar.ax.tick_params(labelsize=18)

# --- ROWS 3 & 4: Zoomed-in Patches (6 total) ---
for i, (y, x) in enumerate(random_origins):
    # NEW: Added +2 instead of +1 so it skips the invisible gap row
    row_idx = 2 + (i // 3)
    col_idx = i % 3

    ax_patch = fig.add_subplot(gs[row_idx, col_idx])

    # Extract patches directly from the reconstructed matrices
    p_bad = recon_bad[y:y+WINDOW_SIZE, x:x+WINDOW_SIZE]
    p_pred = recon_pred[y:y+WINDOW_SIZE, x:x+WINDOW_SIZE]
    p_good = recon_good[y:y+WINDOW_SIZE, x:x+WINDOW_SIZE]


    # Concatenate horizontally: Input | Pred | Target
    combined_patch = np.concatenate((p_bad, p_pred, p_good), axis=1)

    ax_patch.imshow(combined_patch, cmap='seismic', vmin=vmin, vmax=vmax, aspect='auto')

    # Formatting
    ax_patch.set_xticks([])
    ax_patch.set_yticks([])

    # --- NEW: Color the boundary of the patch to match its bounding box ---
    for spine in ax_patch.spines.values():
        spine.set_edgecolor(colors[i])
        spine.set_linestyle('--')
        spine.set_linewidth(4)
    # ----------------------------------------------------------------------

    # Title at bottom
    if row_idx == 3:
      ax_patch.set_xlabel('Input      |         U-Net        |    Target', fontweight='bold', fontsize=26, labelpad=10)
    else:
      ax_patch.set_xlabel(' ', fontweight='bold', fontsize=20, labelpad=10)

    # Add dividing lines
    ax_patch.axvline(x=WINDOW_SIZE, color='black', linewidth=4)
    ax_patch.axvline(x=2 * WINDOW_SIZE, color='black', linewidth=4)
plt.savefig('outputs/figures/generalization_test.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

# ---- PARAMETERS ----
THRESHOLD = 0.01  # same threshold you used earlier
print("Accessing stacked data from in-memory dictionary...")

stack_base_good = final_stacked_datasets['baseline_good_repitability']
stack_mon1_good = final_stacked_datasets['monitoring_stage1_good_repitability']
stack_mon2_good = final_stacked_datasets['monitoring_stage2_good_repitability']

stack_base_bad = final_stacked_datasets['baseline_bad_repitability']
stack_mon1_bad = final_stacked_datasets['monitoring_stage1_bad_repitability']
stack_mon2_bad = final_stacked_datasets['monitoring_stage2_bad_repitability']

good_datasets = [stack_base_good, stack_mon1_good, stack_mon2_good]
bad_datasets = [stack_base_bad, stack_mon1_bad, stack_mon2_bad]
labels = ['Baseline', 'Monitoring Stage 1', 'Monitoring Stage 2']


for i, (bad_data, good_data, label) in enumerate(zip(bad_datasets, good_datasets, labels)):
    print(f"\n🔹 Processing {label}...")

    # --- Normalize and prepare patches ---
    N_data = normalize_data(bad_data.T)
    N_segments_view = sliding_window_view(N_data, window_shape=(WINDOW_SIZE, WINDOW_SIZE), step_size=STEP_SIZE)
    N_segments = N_segments_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()
    N_segments = N_segments[..., np.newaxis]

    # --- Predict and reconstruct ---
    predicted_segments = best_model.predict(N_segments, verbose=0)
    reconstructed_pred = reconstruct_from_patches_average(
        patches=predicted_segments,
        output_shape=N_data.shape,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE
    )

    reconstructed_bad = N_data
    reconstructed_good = normalize_data(good_data.T)

    # --- Compute absolute differences ---
    diff_bad_pred  = np.abs(reconstructed_bad - reconstructed_pred)
    diff_good_pred = np.abs(reconstructed_good - reconstructed_pred)

    # --- Apply threshold mask (only keep > threshold) ---
    masked_diff_bad_pred  = np.where(diff_bad_pred > THRESHOLD, diff_bad_pred, 0)
    masked_diff_good_pred = np.where(diff_good_pred > THRESHOLD, diff_good_pred, 0)

    # --- Compute SSIM and accuracy (below threshold) ---
    ssim_bad_vs_pred = ssim(reconstructed_bad, reconstructed_pred,
                            data_range=reconstructed_bad.max() - reconstructed_bad.min())
    acc_bad_vs_pred  = np.mean(diff_bad_pred <= THRESHOLD) * 100

    ssim_good_vs_pred = ssim(reconstructed_good, reconstructed_pred,
                             data_range=reconstructed_good.max() - reconstructed_good.min())
    acc_good_vs_pred  = np.mean(diff_good_pred <= THRESHOLD) * 100

    print(f"   🟣 Bad vs. Prediction  -> SSIM: {ssim_bad_vs_pred:.4f}, Accuracy (<{THRESHOLD}): {acc_bad_vs_pred:.2f}%")
    print(f"   🟢 Good vs. Prediction -> SSIM: {ssim_good_vs_pred:.4f}, Accuracy (<{THRESHOLD}): {acc_good_vs_pred:.2f}%")

    # --- Visualization ---
    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    titles = [
        "Original (Bad Reliability)",
        "U-Net Reconstructed",
        "Good Reliability",
        "|Bad - Pred| Difference checker ",
        "|Good - Pred| Difference checker ",
        "|Bad - Good| Difference checker"
    ]
    data_list = [
        reconstructed_bad,
        reconstructed_pred,
        reconstructed_good,
        masked_diff_bad_pred,
        masked_diff_good_pred,
        np.abs(reconstructed_bad - reconstructed_good)
    ]

    for ax, dat, title in zip(axs.flat, data_list, titles):
        abs99 = np.percentile(np.abs(dat), 100)
        vmin, vmax = -abs99, abs99
        cmap = 'seismic' if '|' not in title else 'viridis'
        im = ax.imshow(dat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_title(title, fontsize=13)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f"{label}: Prediction vs. Reliability Comparison\n"
                 f"Bad→ SSIM={ssim_bad_vs_pred:.3f}, Acc={acc_bad_vs_pred:.2f}% | "
                 f"Good→ SSIM={ssim_good_vs_pred:.3f}, Acc={acc_good_vs_pred:.2f}%",
                 fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(f"Fig_recon_{label.lower().replace(' ', '_')}.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()


In [ ]:
import numpy as np

def calculate_nrms(data1, data2):
    """
    Calculates the global Normalized Root Mean Square (NRMS) difference
    between two seismic datasets. Outputs as a percentage (%).
    """
    # Calculate Root Mean Square (RMS) for the difference and individual datasets
    rms_diff = np.sqrt(np.mean((data1 - data2)**2))
    rms1 = np.sqrt(np.mean(data1**2))
    rms2 = np.sqrt(np.mean(data2**2))

    # Avoid division by zero
    denominator = rms1 + rms2
    if denominator == 0:
        return 0.0

    # Calculate NRMS percentage
    nrms_percentage = 200.0 * rms_diff / denominator
    return nrms_percentage

# ==========================================
# Inside your processing loop:
# ==========================================

# 1. Calculate NRMS between U-Net Prediction and Good (Target)
nrms_pred_vs_good = calculate_nrms(reconstructed_pred, reconstructed_good)

# 2. Calculate NRMS between Bad (Input) and Good (Target)
nrms_bad_vs_good = calculate_nrms(reconstructed_bad, reconstructed_good)

# Print the results
print(f"   📉 NRMS (Bad vs. Good): {nrms_bad_vs_good:.2f}%")
print(f"   🏆 NRMS (Prediction vs. Good): {nrms_pred_vs_good:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage.metrics import structural_similarity as ssim
from scipy.spatial.distance import cosine

# --- 1. Time-lapse differences and mismatch ---
diff_baseline_gt    = np.abs(reconstructed_basline - reconstructed_groundtruth)
diff_baseline_input = np.abs(reconstructed_basline - reconstructed_input)
diff_baseline_pred = np.abs(reconstructed_basline - reconstructed_prediction)
diff_baseline_pred = np.abs(diff_baseline_pred- diff_gt_pred*1.2)
mismatch            = np.abs(diff_baseline_pred - diff_baseline_gt)

# --- 2. Metrics ---
corr    = np.corrcoef(diff_baseline_pred.ravel(), diff_baseline_gt.ravel())[0, 1]
ssim_val = ssim(diff_baseline_pred, diff_baseline_gt,
                data_range=diff_baseline_gt.max() - diff_baseline_gt.min())
mse     = np.mean((diff_baseline_pred - diff_baseline_gt) ** 2)
cos_sim = 1 - cosine(diff_baseline_pred.ravel(), diff_baseline_gt.ravel())
print(f"Pearson correlation: {corr:.4f}")
print(f"SSIM: {ssim_val:.4f}")
print(f"MSE: {mse:.6f}")
print(f"Cosine similarity: {cos_sim:.4f}")

# --- 3. Shared scales ---
seis_lim  = np.percentile(np.abs(reconstructed_groundtruth), 99)
all_diffs = np.concatenate([diff_baseline_gt.ravel(),
                            diff_baseline_input.ravel(),
                            diff_baseline_pred.ravel()])
threshold = np.percentile(all_diffs, 10)
diff_lim  = np.percentile(all_diffs, 99.5)

diff_gt_t    = np.where(diff_baseline_gt    >= threshold, diff_baseline_gt,    0)
diff_input_t = np.where(diff_baseline_input >= threshold, diff_baseline_input, 0)
diff_pred_t  = np.where(diff_baseline_pred  >= threshold, diff_baseline_pred,  0)

# --- 4. Layout: 6 columns so single panels can be centred ---
fig = plt.figure(figsize=(15, 17))
gs = gridspec.GridSpec(4, 6, height_ratios=[2.0, 1.25, 1.25, 1.6],
                       hspace=0.35, wspace=0.45, figure=fig)

# (a) velocity model, centred
axa = fig.add_subplot(gs[0, 1:5])
ima = axa.imshow(data_2d, aspect='auto', cmap='jet',
                 extent=[0, nx*interval, nz*interval, 0])
axa.set_xlabel('Horizontal Distance (m)', fontsize=13)
axa.set_ylabel('Depth (m)', fontsize=13)
cba = fig.colorbar(ima, ax=axa, fraction=0.035, pad=0.02)
cba.set_label('Vp (m/s)', fontsize=12)
axa.text(-0.08, 1.05, 'a)', transform=axa.transAxes, fontsize=18, fontweight='bold')

# (b) stacked sections, shared seismic scale
row_b = [(gs[1, 0:2], reconstructed_groundtruth, 'Dense Reliability'),
         (gs[1, 2:4], reconstructed_input,       'Original (Scattered Reliability)'),
         (gs[1, 4:6], reconstructed_prediction,  'U-Net Reconstructed')]
axes_b = []
for slot, dat, title in row_b:
    ax = fig.add_subplot(slot)
    imb = ax.imshow(dat, cmap='seismic', vmin=-seis_lim, vmax=seis_lim, aspect='auto')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('CMP Number', fontsize=11)
    axes_b.append(ax)
axes_b[0].set_ylabel('Time Sample', fontsize=11)
cbb = fig.colorbar(imb, ax=axes_b, fraction=0.02, pad=0.02)
cbb.set_label('Amplitude', fontsize=12)
axes_b[0].text(-0.18, 1.12, 'b)', transform=axes_b[0].transAxes, fontsize=18, fontweight='bold')

# (c) time-lapse differences, shared viridis scale
row_c = [(gs[2, 0:2], diff_gt_t,    'Time-lapse\nDense repeatability'),
         (gs[2, 2:4], diff_input_t, 'Time-lapse\nScattered  repeatability'),
         (gs[2, 4:6], diff_pred_t,  'Time-lapse\nScattered repeatability + U-Net')]
axes_c = []
for slot, dat, title in row_c:
    ax = fig.add_subplot(slot)
    imc = ax.imshow(dat, cmap='viridis', vmin=0, vmax=diff_lim, aspect='auto')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('CMP Number', fontsize=11)
    axes_c.append(ax)
axes_c[0].set_ylabel('Time Sample', fontsize=11)
cbc = fig.colorbar(imc, ax=axes_c, fraction=0.02, pad=0.02)
cbc.set_label('Absolute Difference', fontsize=12)
axes_c[0].text(-0.18, 1.18, 'c)', transform=axes_c[0].transAxes, fontsize=18, fontweight='bold')

# (d) mismatch, centred
axd = fig.add_subplot(gs[3, 1:5])
imd = axd.imshow(mismatch, cmap='hot', aspect='auto', vmin=0)
axd.set_xlabel('CMP Number', fontsize=13)
axd.set_ylabel('Time Sample', fontsize=13)
cbd = fig.colorbar(imd, ax=axd, fraction=0.035, pad=0.02)
cbd.set_label(r'$|\Delta d_{U-Net} - \Delta d_{Good}|$', fontsize=12)
axd.text(-0.08, 1.05, 'd)', transform=axd.transAxes, fontsize=18, fontweight='bold')

fig.savefig('Fig7_timelapse_full.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()